# AURORA OMEGA MAX V5.1 ? Purged Rolling-Origin Validation

V5.1 fixes the methodological flaw found in the first V5 run.

The first V5 mechanically passed, but its tune window used `Y-2/Y-1` to validate year `Y`. With a 3Y forward target, those labels would not be known at the decision date. That is selection leakage.

V5.1 uses a purged annual walk-forward protocol:

- validate year `Y`;
- tune model selection and residual blend on `Y-5` and `Y-4` only;
- train the spine and residual models on years `<= Y-6`;
- therefore the latest tune label matures in `Y-1`, before the validation year begins;
- evaluate only on year `Y`.

A pass here is meaningfully stronger than V5. It says AURORA survives rolling-origin validation without lookahead in model selection.


## 1. Runtime and Config


In [ ]:
import os, sys, json, time, random, subprocess, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)

REPO_URL = "https://github.com/tbasaure-sys/fin.git"
REPO_REF = "main"
WORKDIR = Path("/content/fin") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/blsprime_aurora_omega") if IN_COLAB else Path("./_local_data/blsprime_aurora_omega")
PANEL_ROOT = DRIVE_ROOT / "panel"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"

NOTEBOOK_VERSION = "aurora_omega_max_v5_1_purged_rolling_origin_validation"
ARTIFACT_NAME_PREFIX = "omega_v5_1_purged_rolling_origin_validation"
DATA_CUTOFF_DATE = pd.Timestamp.utcnow().tz_localize(None).date().isoformat()
HORIZON_YEARS = 3
TARGET = "ann_return_3y_fwd"
SEED = 7
MIN_TUNE_ROWS = 350
MIN_CORE_ROWS = 1200
MIN_VAL_ROWS = 100
ROLLING_VAL_YEARS = list(range(2013, 2023))

for p in [DRIVE_ROOT, PANEL_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Runtime:", {"python": sys.version.split()[0], "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
print("Drive:", DRIVE_ROOT)
print("Data cutoff:", DATA_CUTOFF_DATE)
print("Validation years:", ROLLING_VAL_YEARS)


## 2. Sync Repo and Imports


In [ ]:
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(WORKDIR)])
    subprocess.check_call(["git", "-C", str(WORKDIR), "fetch", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_REF])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import importlib
import scripts.run_aurora_router_local as router
importlib.reload(router)
from aurora_omega.data import LENS_NAMES

print("Repo:", WORKDIR)
print("Lenses:", LENS_NAMES)


## 3. Rebuild Featured Panel and Harden Universe Filter


In [ ]:
PANEL_CANDIDATES = [
    PANEL_ROOT / "panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_cache_only_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_selfcontained_2005_2024_1500.parquet",
]
panel_path = next((p for p in PANEL_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if panel_path is None:
    raise FileNotFoundError("No raw panel found. Expected one of: " + ", ".join(map(str, PANEL_CANDIDATES)))

panel = pd.read_parquet(panel_path)
print("Loaded raw panel:", panel_path, panel.shape, "tickers:", panel["ticker"].nunique())

featured = router.add_features(panel.copy())
featured = router.add_lens_predictions(featured)
featured["omega_regime"] = featured.apply(router.classify_spine_regime, axis=1)
featured["omega_primary_question"] = featured["omega_regime"].map(router.primary_question_for_regime)
expectations = featured.apply(router.reverse_dcf_expectations, axis=1)
featured["omega_expectations_pressure"] = [e.get("valuation_pressure_score", np.nan) for e in expectations]
featured["omega_feasibility_score"] = [
    router.score_expectation_feasibility(row, e).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]
featured["omega_downside_anchor_score"] = [
    router.anchor_lens_checks(row, router.classify_spine_regime(row), e).get("asset_value", {}).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]

MUST_EXCLUDE_PRODUCT_TICKERS = {
    "ABALX", "FNILX", "VTSAX", "VBTIX", "GBTC", "ETHE", "IBIT", "FBTC", "BITB", "ARKB",
    "SLV", "GLD", "IAU", "USO", "UNG", "SPY", "QQQ", "VOO", "VTI", "IWM", "DIA",
    "TLT", "HYG", "LQD", "BND", "SHY", "IEF", "EEM", "EFA", "XLF", "XLK", "XLE", "XLV",
}
VALID_OPERATING_CANARY_KEEP = {"CVX", "BSX", "BDX", "EQIX", "X"}


def common_operating_equity_mask(frame):
    ticker = frame["ticker"].astype(str).str.upper().str.strip()
    sector = frame.get("sector", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    industry = frame.get("industry", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    name = frame.get("company_name", pd.Series("", index=frame.index)).astype(str).str.lower()
    text = sector + " " + industry + " " + name
    operating_symbol = ticker.str.match(r"^[A-Z]{1,5}([.-][A-Z])?$")
    blank_sector = sector.isin(["", "unknown", "nan", "none"])
    fund_like_text = text.str.contains(
        r"mutual fund|index fund|exchange traded fund|\betf\b|closed-end|target date|money market|portfolio fund|"
        r"open-end|balanced fund|income fund|growth fund|bond fund|large cap fund|small cap fund|"
        r"ishares|vanguard fund|fidelity fund|blackrock fund|bitcoin trust|ethereum trust|grayscale",
        regex=True,
        na=False,
    )
    fund_family_share_class = ticker.str.len().eq(5) & ticker.str.endswith("X")
    known_product_ticker = ticker.isin(MUST_EXCLUDE_PRODUCT_TICKERS)
    return operating_symbol & ~blank_sector & ~fund_like_text & ~fund_family_share_class & ~known_product_ticker

pre_tickers = set(featured["ticker"].astype(str).str.upper())
mask = common_operating_equity_mask(featured)
pre_shape = featured.shape
removed_sample = sorted(set(featured.loc[~mask, "ticker"].astype(str).str.upper()))[:60]
featured = featured.loc[mask].sort_values(["ticker", "year"]).reset_index(drop=True)
post_tickers = set(featured["ticker"].astype(str).str.upper())
filter_audit = {
    "pre_rows": int(pre_shape[0]),
    "post_rows": int(len(featured)),
    "pre_tickers": int(len(pre_tickers)),
    "post_tickers": int(featured["ticker"].nunique()),
    "removed_rows": int(pre_shape[0] - len(featured)),
    "removed_tickers_sample": removed_sample,
    "must_exclude_product_survivors": sorted((MUST_EXCLUDE_PRODUCT_TICKERS & pre_tickers) & post_tickers),
    "wrongly_removed_operating_canaries": sorted((VALID_OPERATING_CANARY_KEEP & pre_tickers) - post_tickers),
}
print(json.dumps(filter_audit, indent=2))
if filter_audit["must_exclude_product_survivors"]:
    raise AssertionError(f"Product/fund canaries survived: {filter_audit['must_exclude_product_survivors']}")
if filter_audit["wrongly_removed_operating_canaries"]:
    raise AssertionError(f"Operating canaries were removed: {filter_audit['wrongly_removed_operating_canaries']}")

display(featured[["ticker", "year", "sector", "industry", "omega_regime", "pred_reverseDcf", "pred_assetValue", TARGET]].head())


## 4. Mature Targets and Common Helpers


In [ ]:
def mask_immature_forward_returns(frame, target_col=TARGET, horizon_years=HORIZON_YEARS, cutoff_date=DATA_CUTOFF_DATE):
    out = frame.copy()
    asof = pd.to_datetime(out["asof_date"], errors="coerce")
    cutoff = pd.Timestamp(cutoff_date)
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_localize(None)
    mature_date = asof + pd.DateOffset(years=horizon_years)
    matured = mature_date.notna() & (mature_date <= cutoff)
    out[f"{target_col}_matured"] = matured
    out.loc[~matured, target_col] = np.nan
    return out


def mae_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    return float(np.mean(np.abs(s["pred"] - s["y"]))) if len(s) else float("nan")


def ic_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 20 or s["pred"].nunique() < 5 or s["y"].nunique() < 5:
        return float("nan")
    return float(s["pred"].rank().corr(s["y"].rank()))


def decile_spread(frame, pred_col, target_col=TARGET):
    spreads = []
    for _, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 80 or sub[pred_col].nunique() < 10:
            continue
        q = pd.qcut(sub[pred_col], 10, labels=False, duplicates="drop")
        if q.max() < 1:
            continue
        spreads.append(float(sub.loc[q == q.max(), target_col].mean() - sub.loc[q == q.min(), target_col].mean()))
    return float(np.mean(spreads)) if spreads else float("nan")


def prior_for_lenses(names):
    raw = []
    for n in names:
        if n == "reverseDcf": raw.append(0.36)
        elif n == "assetValue": raw.append(0.26)
        elif n == "residualIncome": raw.append(0.17)
        elif n == "roicFade": raw.append(0.08)
        elif n == "dcf": raw.append(0.07)
        elif n == "unitEconomics": raw.append(0.04)
        else: raw.append(0.02)
    raw = np.asarray(raw, dtype="float64")
    return raw / raw.sum()


def fit_simplex_spine(frame, lens_cols, target_col=TARGET, epochs=1800, lr=0.05, l2=0.04):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(frame[lens_cols].values.astype("float32"), device=device)
    y = torch.tensor(frame[target_col].values.astype("float32"), device=device)
    prior = torch.tensor(prior_for_lenses([c.replace("pred_", "") for c in lens_cols]).astype("float32"), device=device)
    theta = torch.zeros(len(lens_cols), device=device, requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(epochs):
        w = torch.softmax(theta, dim=0)
        pred = X @ w
        loss = torch.nn.functional.smooth_l1_loss(pred, y, beta=0.04) + l2 * ((w - prior) ** 2).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    with torch.no_grad():
        w = torch.softmax(theta, dim=0).cpu().numpy()
    return {col.replace("pred_", ""): float(weight) for col, weight in zip(lens_cols, w)}


def apply_spine(frame, weights):
    pred = np.zeros(len(frame), dtype="float64")
    for name, weight in weights.items():
        pred += weight * frame[f"pred_{name}"].astype(float).values
    return pred


def make_model_frame(frame):
    forbidden = {TARGET, "year"}
    forbidden_prefixes = ("ann_return_", "price_t", "target_", "future_", "hard_", "omega_weight_", "pred_")
    numeric = []
    for c in frame.columns:
        if c in forbidden:
            continue
        if any(str(c).startswith(p) for p in forbidden_prefixes):
            continue
        if pd.api.types.is_numeric_dtype(frame[c]) and frame[c].notna().sum() >= 50:
            numeric.append(c)
    cat = [c for c in ["omega_regime", "sector", "industry"] if c in frame.columns]
    return numeric, cat


def tune_metrics(pred, y, frame_for_decile=None, pred_col_name="_tmp_pred"):
    out = {"mae": mae_np(pred, y), "ic": ic_np(pred, y)}
    if frame_for_decile is not None:
        tmp = frame_for_decile.copy()
        tmp[pred_col_name] = pred
        out["decile"] = decile_spread(tmp, pred_col_name)
    else:
        out["decile"] = float("nan")
    out["score"] = out["mae"] - 0.030 * (0 if not np.isfinite(out["ic"]) else out["ic"]) - 0.020 * (0 if not np.isfinite(out["decile"]) else out["decile"])
    return out


def tune_blend(anchor, residual_pred, y, frame):
    best = None
    for rho in np.linspace(0.0, 0.80, 33):
        pred = anchor + rho * residual_pred
        row = {"rho": float(rho), **tune_metrics(pred, y, frame)}
        if best is None or row["score"] < best["score"]:
            best = row
    return best


## 5. Prepare Mature Dataset


In [ ]:
data = mask_immature_forward_returns(featured)
for c in [TARGET, "year"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data["year"] = data["year"].astype("Int64")

ACTIVE_3Y_LENSES = [name for name in LENS_NAMES if name != "capitalCycle" and f"pred_{name}" in data.columns]
lens_cols = [f"pred_{name}" for name in ACTIVE_3Y_LENSES]
for c in lens_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna(subset=["ticker", "year", TARGET] + lens_cols).copy()
data["year"] = data["year"].astype(int)

mature_counts = data.groupby("year").size().rename("rows").reset_index()
print("Mature rows:", len(data), "tickers:", data["ticker"].nunique())
display(mature_counts)
print("Active lenses:", ACTIVE_3Y_LENSES)


## 6. Purged Rolling-Origin Fold Runner


In [ ]:
def challenger_library(seed_offset=0):
    seed = SEED + seed_offset * 101
    return {
        "hgb_abs_shallow": HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.035, max_iter=260, max_leaf_nodes=18, l2_regularization=0.08, random_state=seed),
        "hgb_abs_deep": HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.025, max_iter=360, max_leaf_nodes=28, l2_regularization=0.12, random_state=seed + 1),
        "hgb_sq": HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.030, max_iter=260, max_leaf_nodes=18, l2_regularization=0.18, random_state=seed + 2),
        "rf_stable": RandomForestRegressor(n_estimators=260, max_depth=8, min_samples_leaf=18, random_state=seed, n_jobs=-1),
        "rf_smoother": RandomForestRegressor(n_estimators=360, max_depth=6, min_samples_leaf=28, random_state=seed + 3, n_jobs=-1),
        "extra_trees": ExtraTreesRegressor(n_estimators=320, max_depth=7, min_samples_leaf=18, random_state=seed, n_jobs=-1),
        "ridge": Ridge(alpha=8.0, random_state=seed),
        "elastic": ElasticNet(alpha=0.002, l1_ratio=0.10, random_state=seed, max_iter=10000),
        "huber": HuberRegressor(alpha=0.004, epsilon=1.35, max_iter=1200),
    }


def run_fold(frame, val_year):
    # With a 3Y target, labels from year T are only known around T+3.
    # To validate year Y without selection leakage, tune labels must mature before Y begins.
    tune_years = [val_year - HORIZON_YEARS - 2, val_year - HORIZON_YEARS - 1]
    core_end = min(tune_years) - 1
    latest_tune_label_year = max(tune_years) + HORIZON_YEARS
    if latest_tune_label_year >= val_year:
        raise AssertionError(f"Tune labels leak into validation year {val_year}: latest label year {latest_tune_label_year}")
    core = frame[frame["year"] <= core_end].copy()
    tune = frame[frame["year"].isin(tune_years)].copy()
    val = frame[frame["year"] == val_year].copy()
    if len(core) < MIN_CORE_ROWS or len(tune) < MIN_TUNE_ROWS or len(val) < MIN_VAL_ROWS:
        return None, None, {
            "val_year": val_year,
            "skip_reason": "insufficient_rows",
            "core_end": core_end,
            "tune_years": tune_years,
            "latest_tune_label_year": latest_tune_label_year,
            "core_rows": len(core),
            "tune_rows": len(tune),
            "val_rows": len(val),
        }

    spine_weights = fit_simplex_spine(core, lens_cols)
    for fold_frame in [core, tune, val]:
        fold_frame["spine_pred"] = apply_spine(fold_frame, spine_weights)
        fold_frame["uniform_pred"] = fold_frame[lens_cols].mean(axis=1).astype(float)
        lens_values = fold_frame[lens_cols].astype(float)
        fold_frame["lens_mean"] = lens_values.mean(axis=1)
        fold_frame["lens_std"] = lens_values.std(axis=1)
        fold_frame["lens_range"] = lens_values.max(axis=1) - lens_values.min(axis=1)
        fold_frame["reverse_minus_spine"] = fold_frame.get("pred_reverseDcf", fold_frame["spine_pred"]) - fold_frame["spine_pred"]
        fold_frame["asset_minus_spine"] = fold_frame.get("pred_assetValue", fold_frame["spine_pred"]) - fold_frame["spine_pred"]

    feature_cols, cat_cols = make_model_frame(core)
    leakage_prefixes = ("ann_return_", "price_t", "target_", "future_", "hard_", "omega_weight_", "pred_")
    leakage_features = [c for c in feature_cols if c == TARGET or any(str(c).startswith(p) for p in leakage_prefixes)]
    if leakage_features:
        raise AssertionError(f"Fold {val_year} leakage features: {leakage_features}")

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ],
        remainder="drop",
    )

    X_core = core[feature_cols + cat_cols]
    X_tune = tune[feature_cols + cat_cols]
    X_val = val[feature_cols + cat_cols]
    y_core = core[TARGET].astype(float).values
    y_tune = tune[TARGET].astype(float).values
    y_val = val[TARGET].astype(float).values
    resid_core = y_core - core["spine_pred"].astype(float).values

    pred_bank = {}
    single_rows = []
    for name, model in challenger_library(seed_offset=val_year).items():
        pipe = Pipeline([("preprocess", preprocess), ("model", model)])
        pipe.fit(X_core, resid_core)
        tune_resid = pipe.predict(X_tune)
        val_resid = pipe.predict(X_val)
        blend = tune_blend(tune["spine_pred"].astype(float).values, tune_resid, y_tune, tune)
        tune_pred = tune["spine_pred"].astype(float).values + blend["rho"] * tune_resid
        val_pred = val["spine_pred"].astype(float).values + blend["rho"] * val_resid
        pred_bank[name] = {"tune": tune_pred, "val": val_pred}
        m_tune = tune_metrics(tune_pred, y_tune, tune)
        single_rows.append({
            "model": name,
            "rho": blend["rho"],
            "tune_mae": m_tune["mae"],
            "tune_ic": m_tune["ic"],
            "tune_decile": m_tune["decile"],
            "tune_score": m_tune["score"],
            "val_mae": mae_np(val_pred, y_val),
            "val_ic": ic_np(val_pred, y_val),
            "val_decile": decile_spread(val.assign(_p=val_pred), "_p"),
        })
    single_results = pd.DataFrame(single_rows).sort_values("tune_score")

    top_models = list(single_results.head(5)["model"])
    candidate_rows = []
    candidate_bank = {}

    def add_candidate(name, tune_pred, val_pred):
        m_tune = tune_metrics(tune_pred, y_tune, tune)
        row = {
            "model": name,
            "tune_mae": m_tune["mae"],
            "tune_ic": m_tune["ic"],
            "tune_decile": m_tune["decile"],
            "tune_score": m_tune["score"],
            "val_mae": mae_np(val_pred, y_val),
            "val_ic": ic_np(val_pred, y_val),
            "val_decile": decile_spread(val.assign(_p=val_pred), "_p"),
        }
        candidate_rows.append(row)
        candidate_bank[name] = {"tune": tune_pred, "val": val_pred}

    for name in single_results["model"]:
        add_candidate(name, pred_bank[name]["tune"], pred_bank[name]["val"])

    for i, a in enumerate(top_models):
        for b in top_models[i + 1:]:
            best = None
            for wa in np.linspace(0.0, 1.0, 21):
                wb = 1.0 - wa
                tune_pred = wa * pred_bank[a]["tune"] + wb * pred_bank[b]["tune"]
                m = tune_metrics(tune_pred, y_tune, tune)
                if best is None or m["score"] < best["score"]:
                    best = {"wa": float(wa), "wb": float(wb), **m}
            name = f"ens_{a}_{b}_{best['wa']:.2f}_{best['wb']:.2f}"
            add_candidate(
                name,
                best["wa"] * pred_bank[a]["tune"] + best["wb"] * pred_bank[b]["tune"],
                best["wa"] * pred_bank[a]["val"] + best["wb"] * pred_bank[b]["val"],
            )

    for k in [2, 3, 4, 5]:
        names = top_models[:k]
        add_candidate(
            "ens_equal_top" + str(k),
            np.mean([pred_bank[n]["tune"] for n in names], axis=0),
            np.mean([pred_bank[n]["val"] for n in names], axis=0),
        )

    all_results = pd.DataFrame(candidate_rows).sort_values("tune_score")
    champion_name = str(all_results.iloc[0]["model"])
    champion_pred = candidate_bank[champion_name]["val"]
    tune_champion_pred = candidate_bank[champion_name]["tune"]

    agreement_models = top_models[:5]
    tune_stack = np.vstack([pred_bank[n]["tune"] for n in agreement_models]).T
    val_stack = np.vstack([pred_bank[n]["val"] for n in agreement_models]).T
    agreement_cut = float(np.quantile(tune_stack.std(axis=1), 0.60))
    agreement_std = val_stack.std(axis=1)
    high_confidence = agreement_std <= agreement_cut

    val["champion_pred"] = champion_pred
    val["agreement_std"] = agreement_std
    val["high_confidence"] = high_confidence
    val["fold_val_year"] = val_year
    val["fold_champion"] = champion_name

    lens_mae = {name: mae_np(val[f"pred_{name}"], val[TARGET]) for name in ACTIVE_3Y_LENSES}
    best_single_name, best_single_mae = min(lens_mae.items(), key=lambda kv: kv[1])
    fold = {
        "val_year": int(val_year),
        "core_end": int(core_end),
        "tune_years": tune_years,
        "latest_tune_label_year": int(latest_tune_label_year),
        "purged_selection_clean": bool(latest_tune_label_year < val_year),
        "core_rows": int(len(core)),
        "tune_rows": int(len(tune)),
        "val_rows": int(len(val)),
        "core_tickers": int(core["ticker"].nunique()),
        "tune_tickers": int(tune["ticker"].nunique()),
        "val_tickers": int(val["ticker"].nunique()),
        "champion": champion_name,
        "top_models": top_models,
        "spine_weights": spine_weights,
        "feature_count": int(len(feature_cols)),
        "best_single_lens": best_single_name,
        "best_single_mae": float(best_single_mae),
        "champion_mae": mae_np(champion_pred, y_val),
        "spine_mae": mae_np(val["spine_pred"], y_val),
        "uniform_mae": mae_np(val["uniform_pred"], y_val),
        "champion_ic": ic_np(champion_pred, y_val),
        "spine_ic": ic_np(val["spine_pred"], y_val),
        "uniform_ic": ic_np(val["uniform_pred"], y_val),
        "champion_decile_spread": decile_spread(val, "champion_pred"),
        "spine_decile_spread": decile_spread(val, "spine_pred"),
        "high_confidence_rows": int(high_confidence.sum()),
        "high_confidence_mae": mae_np(val.loc[high_confidence, "champion_pred"], val.loc[high_confidence, TARGET]) if high_confidence.any() else float("nan"),
        "high_confidence_ic": ic_np(val.loc[high_confidence, "champion_pred"], val.loc[high_confidence, TARGET]) if high_confidence.any() else float("nan"),
    }
    fold["beats_spine_mae"] = bool(fold["champion_mae"] < fold["spine_mae"] - 0.0005)
    fold["beats_uniform_mae"] = bool(fold["champion_mae"] < fold["uniform_mae"] - 0.0005)
    fold["beats_best_single_mae"] = bool(fold["champion_mae"] < fold["best_single_mae"] - 0.0005)
    fold["positive_ic"] = bool(np.isfinite(fold["champion_ic"]) and fold["champion_ic"] > 0.0)
    fold["positive_decile"] = bool(np.isfinite(fold["champion_decile_spread"]) and fold["champion_decile_spread"] > 0.0)
    return fold, val, {"single_results": single_results, "all_results": all_results}


## 7. Execute Purged Rolling-Origin Validation


In [ ]:
folds = []
prediction_frames = []
skipped = []
start = time.time()
for val_year in ROLLING_VAL_YEARS:
    print(f"Running fold {val_year}...")
    fold, preds, aux = run_fold(data, val_year)
    if fold is None:
        skipped.append(aux)
        print("  skipped", aux)
        continue
    folds.append(fold)
    prediction_frames.append(preds)
    print(" ", {k: fold[k] for k in ["val_year", "champion", "champion_mae", "spine_mae", "uniform_mae", "best_single_mae", "champion_ic", "champion_decile_spread"]})

folds_df = pd.DataFrame(folds)
rolling_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
print("Elapsed minutes:", round((time.time() - start) / 60, 2))
print("Skipped folds:", skipped)
if not folds_df.empty:
    print("Purged label check:", folds_df[["val_year", "core_end", "tune_years", "latest_tune_label_year", "purged_selection_clean"]].to_dict(orient="records"))
display(folds_df)


## 8. Aggregate Scorecard and Purged Gates


In [ ]:
if folds_df.empty:
    raise RuntimeError("No rolling folds completed.")

pooled = rolling_predictions.copy()
summary = {
    "version": NOTEBOOK_VERSION,
    "validation_protocol": "purged_annual_rolling_origin_train_le_y_minus_6_tune_y_minus_5_y_minus_4_validate_y",
    "fold_count": int(len(folds_df)),
    "fold_years": [int(x) for x in folds_df["val_year"].tolist()],
    "total_val_rows": int(len(pooled)),
    "champion_mae_pooled": mae_np(pooled["champion_pred"], pooled[TARGET]),
    "spine_mae_pooled": mae_np(pooled["spine_pred"], pooled[TARGET]),
    "uniform_mae_pooled": mae_np(pooled["uniform_pred"], pooled[TARGET]),
    "champion_ic_pooled": ic_np(pooled["champion_pred"], pooled[TARGET]),
    "spine_ic_pooled": ic_np(pooled["spine_pred"], pooled[TARGET]),
    "uniform_ic_pooled": ic_np(pooled["uniform_pred"], pooled[TARGET]),
    "champion_decile_pooled": decile_spread(pooled, "champion_pred"),
    "spine_decile_pooled": decile_spread(pooled, "spine_pred"),
    "avg_champion_mae": float(folds_df["champion_mae"].mean()),
    "avg_spine_mae": float(folds_df["spine_mae"].mean()),
    "avg_uniform_mae": float(folds_df["uniform_mae"].mean()),
    "avg_champion_ic": float(folds_df["champion_ic"].mean()),
    "beats_spine_mae_share": float(folds_df["beats_spine_mae"].mean()),
    "beats_uniform_mae_share": float(folds_df["beats_uniform_mae"].mean()),
    "beats_best_single_mae_share": float(folds_df["beats_best_single_mae"].mean()),
    "positive_ic_share": float(folds_df["positive_ic"].mean()),
    "positive_decile_share": float(folds_df["positive_decile"].mean()),
    "median_high_confidence_mae": float(folds_df["high_confidence_mae"].median()),
    "model_selection_counts": folds_df["champion"].value_counts().to_dict(),
    "purged_selection_clean_share": float(folds_df["purged_selection_clean"].mean()),
    "filter_audit": filter_audit,
}
summary["mae_lift_vs_spine"] = summary["spine_mae_pooled"] - summary["champion_mae_pooled"]
summary["mae_lift_vs_uniform"] = summary["uniform_mae_pooled"] - summary["champion_mae_pooled"]

gates = {
    "enough_folds": summary["fold_count"] >= 6,
    "enough_rows": summary["total_val_rows"] >= 2500,
    "filter_product_canaries_clean": len(filter_audit["must_exclude_product_survivors"]) == 0,
    "filter_operating_canaries_kept": len(filter_audit["wrongly_removed_operating_canaries"]) == 0,
    "purged_selection_clean": summary["purged_selection_clean_share"] == 1.0,
    "pooled_beats_spine_mae": summary["champion_mae_pooled"] < summary["spine_mae_pooled"] - 0.0020,
    "pooled_beats_uniform_mae": summary["champion_mae_pooled"] < summary["uniform_mae_pooled"] - 0.0020,
    "fold_beats_spine_share": summary["beats_spine_mae_share"] >= 0.70,
    "fold_beats_uniform_share": summary["beats_uniform_mae_share"] >= 0.70,
    "fold_beats_best_single_share": summary["beats_best_single_mae_share"] >= 0.60,
    "pooled_positive_ic": np.isfinite(summary["champion_ic_pooled"]) and summary["champion_ic_pooled"] > 0.025,
    "fold_positive_ic_share": summary["positive_ic_share"] >= 0.60,
    "pooled_positive_decile": np.isfinite(summary["champion_decile_pooled"]) and summary["champion_decile_pooled"] > 0.030,
    "fold_positive_decile_share": summary["positive_decile_share"] >= 0.50,
}
production_candidate = bool(all(gates.values()))
summary["production_candidate"] = production_candidate
summary["product_mode"] = "rolling_origin_validated_memo_router" if production_candidate else "v4_1_candidate_with_rolling_origin_caveats"
summary["gates"] = gates

print(json.dumps({"summary": summary, "gates": gates}, indent=2)[:20000])
display(folds_df[[
    "val_year", "core_end", "tune_years", "latest_tune_label_year", "val_rows", "champion", "champion_mae", "spine_mae", "uniform_mae", "best_single_lens", "best_single_mae", "champion_ic", "champion_decile_spread", "beats_spine_mae", "beats_uniform_mae", "beats_best_single_mae", "positive_ic", "positive_decile"
]])


## 9. Export Purged Artifacts


In [ ]:
out_dir = ARTIFACT_ROOT / f"{ARTIFACT_NAME_PREFIX}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

folds_df.to_csv(out_dir / "rolling_origin_folds.csv", index=False)
rolling_predictions.to_csv(out_dir / "rolling_origin_predictions.csv", index=False)
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(out_dir / "filter_audit.json").write_text(json.dumps(filter_audit, indent=2), encoding="utf-8")
manifest = {
    "version": NOTEBOOK_VERSION,
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "artifact_dir": str(out_dir),
    "data_cutoff_date": DATA_CUTOFF_DATE,
    "summary": summary,
    "gates": gates,
    "decision": "PROMOTE_ROLLING_ORIGIN_VALIDATED_MEMO_ROUTER" if production_candidate else "KEEP_V4_1_AS_CANDIDATE_REQUIRE_MORE_HARDENING",
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2)[:16000])
print("Artifact dir:", out_dir)
